# Real-world Data Wrangling

In this project, you will apply the skills you acquired in the course to gather and wrangle real-world data with two datasets of your choice.

You will retrieve and extract the data, assess the data programmatically and visually, accross elements of data quality and structure, and implement a cleaning strategy for the data. You will then store the updated data into your selected database/data store, combine the data, and answer a research question with the datasets.

Throughout the process, you are expected to:

1. Explain your decisions towards methods used for gathering, assessing, cleaning, storing, and answering the research question
2. Write code comments so your code is more readable

Before you start, install the some of the required packages. 

In [ ]:
!python -m pip install kaggle==1.6.12

: 

In [ ]:
!pip install --target=/workspace ucimlrepo numpy==1.24.3

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

**Note:** Restart the kernel to use updated package(s).

## 1. Gather data

In this section, you will extract data using two different data gathering methods and combine the data. Use at least two different types of data-gathering methods.

### **1.1.** Problem Statement
In 2-4 sentences, explain the kind of problem you want to look at and the datasets you will be wrangling for this project.

Finding the right datasets can be time-consuming. Here we provide you with a list of websites to start with. But we encourage you to explore more websites and find the data that interests you.

* Google Dataset Search https://datasetsearch.research.google.com/
* The U.S. Government’s open data https://data.gov/
* UCI Machine Learning Repository https://archive.ics.uci.edu/ml/index.php


This project investigates how air pollution levels differ across countries and whether highly populated countries tend to have higher average PM2.5 pollution levels.

I will wrangle a world population dataset and an IQAir city-level air quality dataset. 

The population dataset provides country-level population values across multiple years, while the air quality dataset provides city-level PM2.5 values that can be aggregated to country level. 

After cleaning and combining the datasets, I will compare China, India, European countries, and other countries using population and average air pollution indicators.

### **1.2.** Gather at least two datasets using two different data gathering methods

List of data gathering methods:

- Download data manually
- Programmatically downloading files
- Gather data by accessing APIs
- Gather and extract data from HTML files using BeautifulSoup
- Extract data from a SQL database

Each dataset must have at least two variables, and have greater than 500 data samples within each dataset.

For each dataset, briefly describe why you picked the dataset and the gathering method (2-3 full sentences), including the names and significance of the variables in the dataset. Show your work (e.g., if using an API to download the data, please include a snippet of your code). 

Load the dataset programmtically into this notebook.

#### Dataset 1

Type: CSV File

Method:
Programmatically downloaded from a URL using pandas.read_csv().

Dataset:
World Population Dataset (derived from World Bank population statistics).

Dataset variables:

* Country Name: Name of the country or region.
* Country Code: Unique country identifier.
* Year: Observation year.
* Population: Total population for the country in the given year.

Reason for selection:

This dataset contains yearly population values for countries around the world and provides a country-level variable that can be combined with air quality data for analysis. Population was selected because it allows investigation of whether larger populations are associated with higher air pollution levels.

In [ ]:
import pandas as pd

population_url = (
    "https://raw.githubusercontent.com/datasets/population/master/data/population.csv"
)

population_long_raw = pd.read_csv(population_url)

population_raw = population_long_raw.pivot(
    index=["Country Name", "Country Code"],
    columns="Year",
    values="Value"
).reset_index()

population_raw.columns = population_raw.columns.astype(str)

population_raw.head()

In [ ]:
print("Downloaded population shape:", population_long_raw.shape)
print("Reshaped population shape:", population_raw.shape)

In [ ]:
population_long_raw.to_csv(
    "data/raw/world_population_programmatic_raw.csv",
    index=False
)

population_raw.to_csv(
    "data/raw/world_population_reshaped_raw.csv",
    index=False
)

#### Dataset 2

Type: CSV File

Method:
Manually downloaded from Kaggle and loaded locally using pandas.

Dataset:
IQAir Air Quality Index Dataset.

Dataset variables:

* City: City name and country.
* 2021: Average PM2.5 concentration for 2021.
* JAN(2021) - DEC(2021): Monthly PM2.5 measurements.
* Rank: Pollution ranking of the city.

Reason for selection:

This dataset contains PM2.5 air pollution measurements for cities around the world. After aggregating city-level measurements to country level, it can be combined with population data to investigate differences in air pollution and the relationship between population size and pollution levels.

In [ ]:
air_raw = pd.read_csv("data/raw/AIR QUALITY INDEX (by cities) - IQAir.csv")
air_raw.head()

Optional data storing step: You may save your raw dataset files to the local data store before moving to the next step.

In [ ]:
print("Population shape:", population_raw.shape)
print("Air quality shape:", air_raw.shape)

In [ ]:
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/cleaned", exist_ok=True)

population_raw.to_csv("data/raw/world_population_raw.csv", index=False)
air_raw.to_csv("data/raw/iqair_city_air_quality_raw.csv", index=False)

#### Relationship between the datasets

Both datasets contain geographic information that can be linked at the country level. The population dataset provides total population values, while the IQAir dataset provides air pollution measurements. Combining them enables analysis of how population size relates to PM2.5 pollution levels and allows comparison between countries such as China, India, and selected European countries.

## 2. Assess data

Assess the data according to data quality and tidiness metrics using the report below.

List **two** data quality issues and **two** tidiness issues. Assess each data issue visually **and** programmatically, then briefly describe the issue you find.  **Make sure you include justifications for the methods you use for the assessment.**

In [ ]:
population_raw.info()
population_raw.describe()
population_raw.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
air_raw.info()
air_raw.describe()
air_raw.isna().sum().sort_values(ascending=False)

In [ ]:
air_raw.sample(10)

### Quality Issue 1: Missing population values

The population dataset contains missing values in several year columns. This is a completeness issue because missing population values could affect country-level population comparisons and later merging with the air quality dataset.

Visual Assessment:

I displayed the rows containing missing values using population_raw[population_raw.isna().any(axis=1)]. This visual check is useful because it shows which records are affected and helps confirm whether the missing values are limited to a small number of rows.

Programmatic Assessment:

I used population_raw.isna().sum() to count the number of missing values in each column. This programmatic check is appropriate because it quantifies the scale of the missing-data issue and identifies which columns contain missing values.

In [ ]:
population_raw[population_raw.isna().any(axis=1)]

In [ ]:
population_raw.isna().sum().sort_values(ascending=False).head(20)

### Quality Issue 2: Annual PM2.5 columns used in this analysis are stored as incorrect data types

The annual PM2.5 columns used in this analysis, especially 2021, 2020, 2019, 2018, and 2017, are stored as object data types instead of numeric data types. This is a validity issue because these columns represent pollution measurements and must be numeric for calculations, aggregation, and visualization.

Visual Assessment:

I inspected the annual PM2.5 columns using head(). This visual check is appropriate because it shows the actual values in the selected columns and confirms that they contain pollution measurements that should be treated as numbers.

Programmatic Assessment:

I used dtypes to check the data types of the columns. This programmatic check is appropriate because it confirms whether pandas is storing the PM2.5 measurement columns as numeric or non-numeric data.

In [ ]:
air_raw[["2021", "2020", "2019", "2018", "2017"]].head()

In [ ]:
air_raw.dtypes

Justification:

I assessed this issue programmatically using dtypes to check the data type of each column. I also inspected sample rows visually using head() to confirm that the PM2.5 columns contain measurement values that should be numeric.

### Tidiness Issue 1: Years stored as columns

The population dataset is not tidy because years are stored as separate columns from 1960 to 2024. Each year should be a value in a single Year column, with the population stored in a separate Population column.

Visual Assessment:

I inspected the dataset using population_raw.head(). This visual check is appropriate because it shows the wide structure of the dataset, where many year values appear as separate column headers.

Programmatic Assessment:

I used population_raw.columns.tolist() to inspect all column names. This programmatic check is appropriate because it confirms that many columns represent years, which violates the tidy data rule that each variable should form a column.

In [ ]:
# Inspecting the dataframe visually
population_raw.head()

In [ ]:
# Inspecting the dataframe programmatically
population_raw.columns.tolist()

### Tidiness Issue 2: City and country stored together in one column

The air quality dataset is not tidy because the City column contains both city and country in the same field, for example "Bhiwadi, India" and "Hotan, China". City and country are two separate variables, so they should be stored in separate columns.

Visual Assessment:

I inspected the dataset using air_raw.head(). This visual check is appropriate because it shows that the City column contains combined city-country values.

Programmatic Assessment:

I inspected air_raw["City"].head(10). This programmatic check is appropriate because it focuses directly on the affected column and confirms that the same combined pattern appears across multiple rows.

In [ ]:
# Inspecting the dataframe visually
air_raw.head()

In [ ]:
# Inspecting the dataframe programmatically
air_raw["City"].head(10)

## 3. Clean data
Clean the data to solve the 4 issues corresponding to data quality and tidiness found in the assessing step. **Make sure you include justifications for your cleaning decisions.**

After the cleaning for each issue, please use **either** the visually or programatical method to validate the cleaning was succesful.

At this stage, you are also expected to remove variables that are unnecessary for your analysis and combine your datasets. Depending on your datasets, you may choose to perform variable combination and elimination before or after the cleaning stage. Your dataset must have **at least** 4 variables after combining the data.

In [ ]:
population_clean = population_raw.copy()
air_clean = air_raw.copy()

### **Quality Issue 1: Missing population values*

The population dataset contains missing values in several year columns. Missing values can affect calculations and comparisons if not handled appropriately.


Assessment Justification:

I assessed this issue programmatically using isna().sum() to count missing values in each column. I also visually inspected rows containing missing values using filtering to identify which records were affected and to understand the extent of the missing data.

In [ ]:
population_clean = population_clean.dropna()

In [ ]:
population_clean.isna().sum().sum()

Validation:

The result is 0, which confirms that all missing values were successfully removed from the population dataset.

Justification:

I removed rows containing missing population values using dropna() because the missing values were limited to a small number of records. Keeping incomplete records could affect population comparisons and later analysis. After cleaning, I validated the result by checking that the total number of missing values in the dataset was equal to zero.

### **Quality Issue 2: Annual PM2.5 columns used in this analysis are stored as incorrect data types**

The annual PM2.5 columns used in this analysis contain values stored as text, including the placeholder "-" used to represent missing values. This prevents the selected annual PM2.5 columns from being used directly for numerical analysis.


Assessment Justification:

I assessed this issue by inspecting the annual PM2.5 columns used in this analysis and their data types using dtypes. I also examined sample values and identified the presence of "-" placeholders, which indicated that the columns were not being interpreted as purely numeric data.

In [ ]:
# Replace invalid missing-value placeholders with NaN
air_clean = air_clean.replace("-", np.nan)

# Convert PM2.5 columns to numeric data types
pm25_cols = ["2021", "2020", "2019", "2018", "2017"]

air_clean[pm25_cols] = air_clean[pm25_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

In [ ]:
air_clean[pm25_cols].dtypes

In [ ]:
air_clean[pm25_cols].head()

In [ ]:
air_clean[pm25_cols].isna().sum()

Validation:

The selected annual PM2.5 columns used in this analysis are now stored as numeric (float64) data types. The output also confirms that invalid "-" placeholders were successfully converted to NaN values, allowing the columns to be used correctly for analysis and aggregation.

Justification:

I replaced "-" values with NaN because they are invalid placeholders in the annual PM2.5 columns used for this analysis. I then converted the selected annual PM2.5 columns to numeric data types so they can be used for calculations, aggregation, and visualization. I did not convert the monthly PM2.5 columns because they are not used in the final analysis. I validated the cleaning by checking that the selected annual PM2.5 columns are now stored as float64.

### **Tidiness Issue 1: Years stored as columns**

Population years are stored as separate columns rather than observations.

In [ ]:
population_clean = population_clean.melt(
    id_vars=["Country Name", "Country Code"],
    var_name="Year",
    value_name="Population"
)

In [ ]:
population_clean.head()

Justification:

I reshaped the dataset from wide format to long format using melt(). This creates a single Year column and a single Population column, which follows tidy data principles and makes analysis and visualization easier. I validated the transformation by confirming that the resulting dataframe contains the columns Country Name, Country Code, Year, and Population.

### **Tidiness Issue 2: City and country combined**

City and country are stored together in a single column.

In [ ]:
air_clean[["City_Name", "Country"]] = air_clean["City"].str.split(
    ", ",
    n=1,
    expand=True
)

In [ ]:
air_clean[["City", "City_Name", "Country"]].head()

Justification:

I split the City column into separate City_Name and Country columns because they represent two different variables. Separating them improves data organization and simplifies grouping, filtering, and aggregation by country. I validated the transformation by inspecting the new columns and confirming that city and country values were correctly separated.

### **Remove unnecessary variables and combine datasets**

Depending on the datasets, you can also peform the combination before the cleaning steps.

### Keep only 2021 population

Since your air quality data focuses on 2021, extract only 2021 population data.

In [ ]:
population_2021 = population_clean[
    population_clean["Year"] == "2021"
].copy()

population_2021.head()

In [ ]:
population_2021.shape

### Aggregate air quality to country level

Compute average PM2.5 by country.

In [ ]:
country_air = (
    air_clean
    .groupby("Country")["2021"]
    .mean()
    .reset_index()
)

country_air.head()

In [ ]:
country_air.shape

### Rename column for clarity

In [ ]:
country_air.rename(
    columns={"2021": "Avg_PM25_2021"},
    inplace=True
)

### Inspect country names

In [ ]:
sorted(country_air["Country"].unique())[:20]

In [ ]:
sorted(population_2021["Country Name"].unique())[:20]

### Create matching column

In [ ]:
population_2021["Country"] = population_2021["Country Name"]

### Merge

In [ ]:
combined_df = pd.merge(
    population_2021,
    country_air,
    on="Country",
    how="inner"
)

In [ ]:
combined_df = combined_df[
    ["Country", "Country Code", "Year", "Population", "Avg_PM25_2021"]
]


### Validate

In [ ]:
combined_df.head()

In [ ]:
combined_df.shape

In [ ]:
print("Population countries in 2021:", population_2021["Country"].nunique())
print("Air quality countries:", country_air["Country"].nunique())
print("Countries after merge:", combined_df["Country"].nunique())

Merge Quality Check:

This check shows how many unique countries existed in each dataset before the merge and how many countries remained after the inner merge. It helps identify whether country-name mismatches caused records to be excluded from the final combined dataset.

In [ ]:
combined_df.columns

Validation:

The final combined dataset contains 97 rows and 5 columns. The first rows show that country, country code, year, population, and average PM2.5 values were successfully combined.

Justification:

To answer the research question, I needed both datasets at the same country-year level. I filtered the population dataset to 2021, aggregated city-level PM2.5 values to country-level averages, merged both datasets by country, and removed duplicate or unnecessary variables. The final combined dataset contains more than four variables and is ready for storage and analysis.

## 4. Update your data store
Update your local database/data store with the cleaned data, following best practices for storing your cleaned data:

- Must maintain different instances / versions of data (raw and cleaned data)
- Must name the dataset files informatively
- Ensure both the raw and cleaned data is saved to your database/data store

In [ ]:
# Save cleaned population dataset
population_clean.to_csv(
    "data/cleaned/population_clean.csv",
    index=False
)

# Save cleaned air quality dataset
air_clean.to_csv(
    "data/cleaned/air_quality_clean.csv",
    index=False
)

# Save final combined dataset
combined_df.to_csv(
    "data/cleaned/combined_population_air_quality.csv",
    index=False
)

In [ ]:
# Confirm the file was saved
pd.read_csv("data/cleaned/combined_population_air_quality.csv").head()

The cleaned population dataset, cleaned air quality dataset, and final combined dataset were saved in the cleaned data folder. The original raw files remain stored separately in the raw data folder, maintaining separate raw and cleaned versions of the data. This follows good data management practices and allows the cleaning process to be reproduced if needed.

## 5. Answer the research question

### **5.1:** Define and answer the research question 
Going back to the problem statement in step 1, use the cleaned data to answer the question you raised. Produce **at least** two visualizations using the cleaned data and explain how they help you answer the question.

*Research question: How do China, India, and European countries compare in terms of average PM2.5 air pollution, and is population size related to pollution levels?

In [ ]:
df = combined_df.copy()

In [ ]:
countries = [
    "China",
    "India",
    "Germany",
    "France",
    "Italy",
    "Spain"
]

comparison_df = df[
    df["Country"].isin(countries)
]

comparison_df = comparison_df.sort_values(
    "Avg_PM25_2021",
    ascending=False
)

plt.figure(figsize=(8,5))
plt.bar(
    comparison_df["Country"],
    comparison_df["Avg_PM25_2021"]
)

plt.title("Average PM2.5 Pollution in Selected Countries (2021)")
plt.xlabel("Country")
plt.ylabel("Average PM2.5")

plt.xticks(rotation=45)

plt.show()

*Answer to research question:This visualization compares average PM2.5 pollution levels across China, India, and selected European countries. The chart shows that India and China have significantly higher PM2.5 levels than Germany, France, Italy, and Spain. This suggests that air pollution is generally more severe in China and India than in the selected European countries.

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    comparison_df["Country"],
    comparison_df["Avg_PM25_2021"]
)

plt.title("Average PM2.5 Pollution in Selected Countries (2021)")
plt.xlabel("Country")
plt.ylabel("Average PM2.5")
plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    "images/pollution_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    df["Population"],
    df["Avg_PM25_2021"]
)

plt.title("Population vs Average PM2.5 Pollution")
plt.xlabel("Population")
plt.ylabel("Average PM2.5")

plt.tight_layout()

plt.savefig(
    "images/population_vs_pm25.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

*Answer to research question: This scatter plot explores the relationship between population size and average PM2.5 pollution. Although some highly populated countries have high pollution levels, the points are widely dispersed and do not form a strong pattern. This indicates that population size alone does not fully explain differences in pollution levels across countries.

### **5.2:** Reflection
In 2-4 sentences, if you had more time to complete the project, what actions would you take? For example, which data quality and structural issues would you look into further, and what research questions would you further explore?

*Answer:* If I had more time, I would investigate country name inconsistencies to improve the merge between datasets and increase the number of matched countries. I would also explore additional variables such as GDP, industrial activity, and energy consumption to better understand the factors influencing air pollution. Finally, I would perform a more detailed statistical analysis to quantify the relationship between population size and PM2.5 pollution levels.